# 08 Multiple Linear Regression

This notebook fits a full multiple linear regression model with all encoded predictors, evaluates it on the test set, and reports the main regression diagnostics.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm

train_path = Path("../data/processed/ElectraHub_train_encoded.csv")
test_path = Path("../data/processed/ElectraHub_test_encoded.csv")

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

target_col = "Sales"
feature_cols = [col for col in train_df.columns if col != target_col]

X_train = sm.add_constant(train_df[feature_cols].astype(float), has_constant="add")
y_train = train_df[target_col]
X_test = sm.add_constant(test_df[feature_cols].astype(float), has_constant="add")
y_test = test_df[target_col]

model = sm.OLS(y_train, X_train).fit()
test_predictions = model.predict(X_test)

def adjusted_r2(r2, n_obs, n_predictors):
    return 1 - (1 - r2) * (n_obs - 1) / (n_obs - n_predictors - 1)

train_r2 = model.rsquared
train_adj_r2 = model.rsquared_adj
test_r2 = 1 - np.sum((y_test - test_predictions) ** 2) / np.sum((y_test - y_test.mean()) ** 2)
test_adj_r2 = adjusted_r2(test_r2, len(y_test), len(feature_cols))
rmse = np.sqrt(np.mean((y_test - test_predictions) ** 2))
mae = np.mean(np.abs(y_test - test_predictions))

print(f"Training rows: {len(train_df)}")
print(f"Test rows: {len(test_df)}")
print(f"Predictors used: {len(feature_cols)}")
print()
print(f"Train R^2: {train_r2:.4f}")
print(f"Train Adjusted R^2: {train_adj_r2:.4f}")
print(f"Train F-statistic: {model.fvalue:.4f}")
print()
print(f"Test R^2: {test_r2:.4f}")
print(f"Test Adjusted R^2: {test_adj_r2:.4f}")
print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")


Training rows: 2400
Test rows: 600
Predictors used: 19

Train R^2: 0.9099
Train Adjusted R^2: 0.9092
Train F-statistic: 1264.5360

Test R^2: 0.9124
Test Adjusted R^2: 0.9095
Test RMSE: 2143.0642
Test MAE: 1728.9721


In [2]:
coef_table = pd.DataFrame(
    {
        "term": model.params.index,
        "coefficient": model.params.values,
        "p_value": model.pvalues.values,
    }
)
coef_table["abs_coefficient"] = coef_table["coefficient"].abs()
coef_table = coef_table.sort_values("abs_coefficient", ascending=False).drop(columns="abs_coefficient")

pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

print(coef_table.to_string(index=False))


                      term   coefficient  p_value
                     const -44139.997311 0.000000
    Competitor_Price_Index -10821.374190 0.000000
       Avg_Customer_Rating  10416.193280 0.000000
                Popularity   1539.868165 0.000000
               Return_Rate  -1530.800904 0.000000
              Region_North  -1013.802986 0.000000
  Campaign_Type_Influencer    937.340507 0.000000
              Region_South   -764.017523 0.000000
Campaign_Type_Social Media    715.509363 0.000000
   Campaign_Type_Search Ad    443.616269 0.000442
   Product_Category_Tablet    429.644303 0.000017
       Discount_Percentage    291.694603 0.000000
               Region_West    250.071565 0.032247
 Campaign_Engagement_Score    162.974181 0.000000
        Product_Age_Months    -90.632015 0.000000
   Advertising_Expenditure     40.599455 0.000000
             Product_Price     -2.340340 0.000000
Length_Product_Description      0.281212 0.680142
           Inventory_Level      0.243832 0.108214


## Assumption Checks and VIF

This section checks common OLS assumptions on the training residuals and reports variance inflation factors for the predictors.

In [3]:
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan, het_white, linear_reset
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson, jarque_bera

resid = model.resid

bp_lm, bp_lm_pvalue, bp_fvalue, bp_f_pvalue = het_breuschpagan(resid, model.model.exog)
white_lm, white_lm_pvalue, white_fvalue, white_f_pvalue = het_white(resid, model.model.exog)
jb_stat, jb_pvalue, jb_skew, jb_kurtosis = jarque_bera(resid)
omnibus = sm.stats.stattools.omni_normtest(resid)
shapiro_sample = resid.sample(500, random_state=0) if len(resid) > 500 else resid
shapiro_stat, shapiro_pvalue = stats.shapiro(shapiro_sample)
dw = durbin_watson(resid)
reset = linear_reset(model, power=2, use_f=True)

vif_table = pd.DataFrame(
    {
        "predictor": X_train.columns,
        "VIF": [variance_inflation_factor(X_train.values, i) for i in range(X_train.shape[1])],
    }
)
vif_table = vif_table[vif_table["predictor"] != "const"].sort_values("VIF", ascending=False)

print("Assumption Checks")
print(f"Durbin-Watson: {dw:.4f}")
print(f"Breusch-Pagan LM: {bp_lm:.4f} (p={bp_lm_pvalue:.6g})")
print(f"Breusch-Pagan F: {bp_fvalue:.4f} (p={bp_f_pvalue:.6g})")
print(f"White LM: {white_lm:.4f} (p={white_lm_pvalue:.6g})")
print(f"White F: {white_fvalue:.4f} (p={white_f_pvalue:.6g})")
print(f"Jarque-Bera: {jb_stat:.4f} (p={jb_pvalue:.6g})")
print(f"Omnibus: {omnibus.statistic:.4f} (p={omnibus.pvalue:.6g})")
print(f"Shapiro-Wilk (sampled residuals): {shapiro_stat:.4f} (p={shapiro_pvalue:.6g})")
print(f"Ramsey RESET F: {reset.fvalue:.4f} (p={reset.pvalue:.6g})")
print()
print("VIF Table")
print(vif_table.to_string(index=False))


Assumption Checks
Durbin-Watson: 1.9966
Breusch-Pagan LM: 22.5277 (p=0.258787)
Breusch-Pagan F: 1.1869 (p=0.258889)
White LM: 232.8694 (p=0.0367312)
White F: 1.2078 (p=0.0309203)
Jarque-Bera: 9.2266 (p=0.0099192)
Omnibus: 8.0009 (p=0.0183072)
Shapiro-Wilk (sampled residuals): 0.9971 (p=0.538978)
Ramsey RESET F: 23.8653 (p=1.10184e-06)

VIF Table
                 predictor      VIF
                Popularity 2.062831
Campaign_Type_Social Media 1.890406
 Campaign_Engagement_Score 1.815082
  Campaign_Type_Influencer 1.804335
   Campaign_Type_Search Ad 1.764223
               Num_Reviews 1.585638
               Region_West 1.488535
              Region_North 1.434403
              Region_South 1.431929
        Product_Age_Months 1.412183
       Avg_Customer_Rating 1.270676
   Product_Category_Tablet 1.247620
               Return_Rate 1.224150
             Product_Price 1.210263
       Discount_Percentage 1.208110
   Advertising_Expenditure 1.168265
           Inventory_Level 1.047183
    

## Final Model Comparison

This table compares the naive baseline, the three simple linear regression models, and the full multiple linear regression model on the test set.

In [4]:
def test_adjusted_r2(r2, n_obs, n_predictors):
    return 1 - (1 - r2) * (n_obs - 1) / (n_obs - n_predictors - 1)

def evaluate_model(y_true, y_pred, train_r2, n_predictors):
    residuals = y_true - y_pred
    r2 = 1 - np.sum(residuals ** 2) / np.sum((y_true - y_true.mean()) ** 2)
    adj_r2 = test_adjusted_r2(r2, len(y_true), n_predictors)
    rmse = np.sqrt(np.mean(residuals ** 2))
    mae = np.mean(np.abs(residuals))
    return r2, adj_r2, rmse, mae

multiple_lr_rmse = rmse
multiple_lr_mae = mae

# Naive baseline: predict the training mean of Sales for every test row.
naive_prediction = train_df[target_col].mean()
naive_predictions = np.repeat(naive_prediction, len(test_df))
naive_residuals = test_df[target_col] - naive_predictions
naive_r2 = 1 - np.sum(naive_residuals ** 2) / np.sum((test_df[target_col] - test_df[target_col].mean()) ** 2)
naive_rmse = np.sqrt(np.mean(naive_residuals ** 2))
naive_mae = np.mean(np.abs(naive_residuals))
naive_adj_r2 = np.nan

# Refit the three simple LR models so this notebook is self-contained.
simple_models = [
    ("Campaign_Engagement_Score", "Campaign_Engagement_Score"),
    ("Advertising_Expenditure", "Advertising_Expenditure"),
    ("Avg_Customer_Rating", "Avg_Customer_Rating"),
]

comparison_rows = [
    {
        "Model": "Naive baseline",
        "Predictor(s)": "Train mean of Sales",
        "R2": naive_r2,
        "Adjusted R2": naive_adj_r2,
        "RMSE": naive_rmse,
        "MAE": naive_mae,
    }
]

for label, feature in simple_models:
    x_train = sm.add_constant(train_df[[feature]].astype(float), has_constant="add")
    y_train = train_df[target_col]
    x_test = sm.add_constant(test_df[[feature]].astype(float), has_constant="add")
    y_test = test_df[target_col]
    simple_model = sm.OLS(y_train, x_train).fit()
    predictions = simple_model.predict(x_test)
    residuals = y_test - predictions
    r2 = 1 - np.sum(residuals ** 2) / np.sum((y_test - y_test.mean()) ** 2)
    adj_r2 = test_adjusted_r2(r2, len(y_test), 1)
    rmse = np.sqrt(np.mean(residuals ** 2))
    mae = np.mean(np.abs(residuals))

    comparison_rows.append(
        {
            "Model": "Simple LR",
            "Predictor(s)": label,
            "R2": r2,
            "Adjusted R2": adj_r2,
            "RMSE": rmse,
            "MAE": mae,
        }
    )

comparison_rows.append(
    {
        "Model": "Multiple LR",
        "Predictor(s)": "All encoded predictors",
        "R2": test_r2,
        "Adjusted R2": test_adj_r2,
        "RMSE": multiple_lr_rmse,
        "MAE": multiple_lr_mae,
    }
)

comparison_table = pd.DataFrame(comparison_rows)
comparison_table[["R2", "Adjusted R2", "RMSE", "MAE"]] = comparison_table[["R2", "Adjusted R2", "RMSE", "MAE"]].round(4)

comparison_table


,Model,Predictor(s),R2,Adjusted R2,RMSE,MAE
0,Naive baseline,Train mean of Sales,-0.002600,NaN,7249.987700,5818.216500
1,Simple LR,Campaign_Engagement_Score,0.523200,0.522400,4999.375100,3992.425000
2,Simple LR,Advertising_Expenditure,0.446800,0.445900,5385.185000,4399.907000
3,Simple LR,Avg_Customer_Rating,0.196200,0.194900,6491.252900,5124.708100
4,Multiple LR,All encoded predictors,0.912400,0.909500,2143.064200,1728.972100
